# 03A 多模型训练：回归与分类不等于 Random Forest

> 🟢 **Level A · 必须掌握** | 完成标准：在同一数据、同一 split、同一 metrics 下比较多个模型。


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline


## 为什么比较这些模型？
Linear/Ridge 建立加性基线；KNN 依赖距离与 scaling；浅树展示分段规则；随机森林对多棵树取平均；boosting 逐步修正误差。使用同一批行与指标，先和“总预测训练集均值”比较。此处单次测试排名用于学习，最终模型选择应采用 03B/04A 的开发集 CV。


In [ ]:
from sklearn.dummy import DummyRegressor, DummyClassifier
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression,Ridge,LogisticRegression
from sklearn.neighbors import KNeighborsRegressor,KNeighborsClassifier
from sklearn.tree import DecisionTreeRegressor,DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor,ExtraTreesRegressor,GradientBoostingRegressor,RandomForestClassifier,GradientBoostingClassifier
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score,accuracy_score,f1_score
df=pd.read_csv('https://raw.githubusercontent.com/gokhanonderaksu/COFSpace/main/OnlyCoRECOF%20-%20Feature%20Sets/CoRECOF%20-%20CO2%20-%201%20BAR.csv')
features=['PLD (Å)','LCD (Å)','Sacc (m2/g-1)','Porosity','%C','%H','%N','%O','%Metalloid','%Halogen','%Ametal']
X=df[features].copy(); y=df['CO2-1 bar (mol/kg)']


In [ ]:
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.2,random_state=42)
models={'Mean baseline':DummyRegressor(strategy='mean'),'Linear':make_pipeline(StandardScaler(),LinearRegression()),'Ridge':make_pipeline(StandardScaler(),Ridge()),'KNN':make_pipeline(StandardScaler(),KNeighborsRegressor(7)),'Decision Tree':DecisionTreeRegressor(max_depth=8,random_state=42),'Random Forest':RandomForestRegressor(n_estimators=300,random_state=42,n_jobs=-1),'Extra Trees':ExtraTreesRegressor(n_estimators=300,random_state=42,n_jobs=-1),'Gradient Boosting':GradientBoostingRegressor(random_state=42)}
rows=[]
for name,m in models.items():
    m=make_pipeline(SimpleImputer(strategy="median"), m)
    m.fit(Xtr,ytr); p=m.predict(Xte); rows.append([name,mean_absolute_error(yte,p),mean_squared_error(yte,p)**0.5,r2_score(yte,p)])
display(pd.DataFrame(rows,columns=['Model','MAE','RMSE','R2']).sort_values('MAE'))


In [ ]:
Xtr,Xte,ytr_cont,yte_cont=train_test_split(X,y,test_size=0.2,random_state=42)
threshold=ytr_cont.median()
ytr=(ytr_cont>=threshold).astype(int); yte=(yte_cont>=threshold).astype(int)
models={'Majority baseline':DummyClassifier(strategy='most_frequent'),'Logistic':make_pipeline(StandardScaler(),LogisticRegression(max_iter=2000)),'KNN':make_pipeline(StandardScaler(),KNeighborsClassifier(7)),'Decision Tree':DecisionTreeClassifier(max_depth=6,random_state=42),'Random Forest':RandomForestClassifier(n_estimators=300,random_state=42,n_jobs=-1),'Gradient Boosting':GradientBoostingClassifier(random_state=42)}
rows=[]
for name,m in models.items():
    m=make_pipeline(SimpleImputer(strategy="median"), m)
    m.fit(Xtr,ytr); p=m.predict(Xte); rows.append([name,accuracy_score(yte,p),f1_score(yte,p)])
display(pd.DataFrame(rows,columns=['Model','Accuracy','F1']).sort_values('F1',ascending=False))


### 03A 完成标准
知道最高单次分数不是唯一标准，还要比较稳定性、train/test gap、解释性与科学目标。


## 分类分数怎么读？
中位数标签是教学定义，不是工程指标。accuracy 统计总体正确率；precision 关注选中的候选有多少真阳性；recall 关注真正的高吸附材料找回了多少。检查类别比例和 F1，并与多数类基线比较。阈值应预先规定，或仅由训练数据确定。


本例的中位数只从训练部分计算，再冻结用于测试标签。改变阈值意味着改变分类任务，不能只按测试分数选阈值。


## 数据来源与扩展阅读
[Dataset contracts / 数据使用约定](../docs/data_resources.md) · [COFSpace](https://github.com/gokhanonderaksu/COFSpace) · [CURATED-COFs](https://github.com/danieleongari/CURATED-COFs)
